# Text as Features
Accompanies the notes on **Text as Features**.

## 1. Bag of words and TF-IDF by hand

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["the card was charged twice", "the late fee was charged", "card stolen card blocked"]
counts = CountVectorizer()
X = counts.fit_transform(docs)
print(pd.DataFrame(X.toarray(), columns=counts.get_feature_names_out(), index=["d1", "d2", "d3"]))
tfidf = TfidfVectorizer()
T = tfidf.fit_transform(docs)
print("idf:", {w: round(float(v), 4) for w, v in zip(tfidf.get_feature_names_out(), tfidf.idf_)})
print(pd.DataFrame(T.toarray(), columns=tfidf.get_feature_names_out(), index=["d1", "d2", "d3"]).round(3))

## 2. Cleaning: lowercase, punctuation, stopwords, stemming

In [ ]:
import re
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stemmer = PorterStemmer()

def clean(text, remove_stopwords=True, stem=True):
    text = text.lower()
    text = re.sub(r"x{2,}", " ", text)                  # the complaints mask private details as XXXX
    text = re.sub(r"[^a-z\s]", " ", text)               # keep letters only: drops punctuation and digits
    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

example = "I was charged a late fee of $35.00 on XX/XX/2015, although my payments were made on time!"
print("original      :", example)
print("letters only  :", clean(example, remove_stopwords=False, stem=False))
print("no stopwords  :", clean(example, stem=False))
print("stemmed       :", clean(example))
print("negations in the stopword list:", [w for w in ["not", "no", "nor", "never", "cannot"] if w in ENGLISH_STOP_WORDS])
print([stemmer.stem(w) for w in ["payment", "payments", "paying", "paid", "university", "universe", "studies", "studying"]])

## 3. How cleaning shrinks the vocabulary

In [ ]:
cc = pd.read_csv("../../data/Consumer_Complaints_train.csv.zip", usecols=["Product", "Consumer complaint narrative"]).dropna()
cc = cc.sample(20000, random_state=0)
texts = cc["Consumer complaint narrative"]
for name, kwargs in [("raw (letters only)", dict(remove_stopwords=False, stem=False)),
                     ("no stopwords", dict(stem=False)), ("no stopwords + stemmed", dict())]:
    cleaned = texts.map(lambda t: clean(t, **kwargs))
    v = CountVectorizer().fit(cleaned)
    v5 = CountVectorizer(min_df=5).fit(cleaned)
    words = cleaned.str.split().map(len).sum()
    print(f"{name:24s}: {words:8d} words in total, vocabulary {len(v.vocabulary_):6d}, words in >= 5 documents {len(v5.vocabulary_):6d}")

## 4. Counts vs TF-IDF vs bigrams for classifying complaints

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

cleaned = texts.map(clean)
x_train, x_test, y_train, y_test = train_test_split(cleaned, cc["Product"], test_size=0.25, random_state=0, stratify=cc["Product"])
setups = {"counts, unigrams": CountVectorizer(min_df=3),
          "TF-IDF, unigrams": TfidfVectorizer(min_df=3),
          "TF-IDF, unigrams + bigrams": TfidfVectorizer(min_df=3, ngram_range=(1, 2))}
for name, vect in setups.items():
    for mname, model in [("logistic", LogisticRegression(max_iter=3000, C=5)), ("multinomial NB", MultinomialNB(alpha=0.1))]:
        pipe = make_pipeline(vect, model).fit(x_train, y_train)
        print(f"{name:27s} {mname:15s}: test accuracy {pipe.score(x_test, y_test):.3f}, features {len(pipe[0].vocabulary_)}")

## 5. Cosine similarity: a tiny search engine

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

vect = TfidfVectorizer(min_df=3).fit(x_train)
M = vect.transform(x_train)
query = vect.transform([clean("my student loan servicer will not let me change my repayment plan")])
sims = cosine_similarity(query, M).ravel()
for i in sims.argsort()[::-1][:3]:
    print(f"cosine {sims[i]:.3f} | {y_train.iloc[i]:15s} | {x_train.iloc[i][:90]}")

## 6. Why cosine and not Euclidean distance on counts

In [ ]:
a = "late fee charged"
b = " ".join(["late fee charged"] * 10)                  # same words, ten times as long
c = "card stolen blocked"
C = CountVectorizer().fit([a, b, c]).transform([a, b, c])
print("Euclidean on counts: d(a,b) =", euclidean_distances(C[0], C[1]).item().__round__(3), " d(a,c) =", euclidean_distances(C[0], C[2]).item().__round__(3))
print("cosine similarity  : s(a,b) =", cosine_similarity(C[0], C[1]).item().__round__(3), " s(a,c) =", cosine_similarity(C[0], C[2]).item().__round__(3))

## Try it yourself
1. Repeat section 4 with `clean(t, stem=False)`.
2. Cosine similarity of d1 and d2 in section 1.